# 05 - Audio to Text Response

This notebook transcribes a customer voice message and runs sentiment/urgency triage on the transcript.

Pre-requisite: run `04_image_test.ipynb` first.

---

## Step 1 – Load environment variables

In [1]:
import json
import os
from pathlib import Path

import requests
import azure.cognitiveservices.speech as speechsdk
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env")

endpoint = os.environ["AZURE_AI_ENDPOINT"].rstrip("/")
deployment = os.environ["AZURE_AI_DEPLOYMENT"]
location = os.environ["AZURE_LOCATION"]
auth_mode = os.environ.get("AZURE_AUTH_MODE", "aad").lower()
if auth_mode != "aad":
    raise RuntimeError(f"Unsupported auth mode for this demo: {auth_mode}. Expected 'aad'.")

credential = AzureCliCredential()
credential.get_token("https://cognitiveservices.azure.com/.default")
credential_source = "AzureCliCredential"
audio_path = Path(os.environ["LOCAL_AUDIO_PATH"]).expanduser()

if not audio_path.exists():
    raise FileNotFoundError(f"Audio file not found: {audio_path}")

output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

print(f"Audio path        : {audio_path.resolve()}")
print(f"Credential source : {credential_source}")

Audio path        : C:\Users\hannahhowell\OneDrive - Microsoft\Documents\Git\azure-solution-prototypes\demos\gpt4o01-text-image-audio\data\goodicecream.wav
Credential source : AzureCliCredential


## Step 2 – Create clients / connections

Initialise any SDK clients using the loaded environment variables.

In [2]:
speech_token = credential.get_token("https://cognitiveservices.azure.com/.default").token
speech_endpoint = os.environ["AZURE_AI_ENDPOINT"]

speech_config = speechsdk.SpeechConfig(endpoint=speech_endpoint)
speech_config.authorization_token = speech_token
speech_config.speech_recognition_language = "en-US"
audio_config = speechsdk.audio.AudioConfig(filename=str(audio_path))
recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)

speech_result = recognizer.recognize_once_async().get()
if speech_result.reason != speechsdk.ResultReason.RecognizedSpeech:
    raise RuntimeError(f"Speech recognition failed: {speech_result.reason}")

transcript = speech_result.text
print("Transcript:")
print(transcript)

Transcript:
Mooth, texture, balance, sweetness and justice. The right level of chill. It quietly does everything you'd want without trying too hard. The kind of thing you keep going back to, not out of habit, but because it genuinely makes the moment better.


## Step 3 – Demo scenario

Implement the key scenario that validates the customer hypothesis.

In [5]:
url = f"{endpoint}/openai/deployments/{deployment}/chat/completions?api-version=2024-10-21"
messages = [
    {"role": "system", "content": "You are a concise sentiment triage assistant."},
    {"role": "user", "content": f"Using this customer transcript, return speech to text, sentiment (positive/neutral/negative), urgency (low/medium/high), and one recommended next action: {transcript}"},
]

model_token = credential.get_token("https://cognitiveservices.azure.com/.default").token
response = requests.post(
    url,
    headers={"Authorization": f"Bearer {model_token}", "Content-Type": "application/json"},
    json={"messages": messages, "max_completion_tokens": 250},
    timeout=90,
)
if response.status_code >= 400:
    raise RuntimeError(f"GPT call failed: {response.status_code} {response.text}")

body = response.json()
text_response = body["choices"][0]["message"]["content"]
print("Model response:")
print(text_response)

Model response:
**Speech to Text:**  
Mooth, texture, balance, sweetness and justice. The right level of chill. It quietly does everything you'd want without trying too hard. The kind of thing you keep going back to, not out of habit, but because it genuinely makes the moment better.

**Sentiment:**  
Positive  

**Urgency:**  
Low  

**Recommended Next Action:**  
Acknowledge the positive feedback with gratitude and consider leveraging this sentiment in promotional or testimonial materials.


---

## Audio to text checkpoint complete

Next: run **06_audio_roundtrip.ipynb** for speech-in to speech-out validation.